# 03 -- Sensibilidad a tamaño de flota: H0 vs H1 vs H2 vs H3

Barre el número de barcos manteniendo todo lo demás constante. Rango
extendido a **4-16 barcos** (antes llegaba hasta 12) para ver si la curva de
servicio sigue mejorando o ya se aplana (saturación) con esta demanda.
Además del servicio (espera, tiempo en sistema), se agrega una vista de
**cuántos barcos están en movimiento vs. quietos en cada minuto del día**
-- para responder si, en horas valle, hace falta tener toda la flota
desplegada o se podría operar con menos.

Todas las gráficas son interactivas (Plotly: zoom, pan, hover con el valor
exacto, clic en la leyenda para aislar una serie).

In [1]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

BASE = Path.cwd().parent.parent
sys.path.insert(0, str(BASE / "simulador" / "src"))
sys.path.insert(0, str(BASE / "politica_base" / "src"))
sys.path.insert(0, str(BASE / "heuristicas" / "comun" / "src"))
sys.path.insert(0, str(BASE / "heuristicas" / "h1_reserva" / "src"))
sys.path.insert(0, str(BASE / "heuristicas" / "h2_costo_local" / "src"))
sys.path.insert(0, str(BASE / "heuristicas" / "h3_costo_global" / "src"))

from env import SimuladorBarcosBergen
from politica_base import asignar_flota
from politica_h1 import asignar_flota_h1
from politica_h2 import asignar_flota_h2
from politica_h3 import asignar_flota_h3
import metricas as met

cfg_sim = yaml.safe_load(open(BASE / "simulador" / "config" / "instance.yaml", encoding="utf-8"))
cfg_bb = yaml.safe_load(open(BASE / "bergen-boats" / "config" / "instance.yaml", encoding="utf-8"))
nodos = [n["id"] for n in cfg_bb["nodos_demanda"]]
matriz_tiempos = pd.read_csv(BASE / "bergen-boats" / "02_ruteo_navegable" / "output" / "matriz_tiempos_min.csv", index_col=0)
capacidad = cfg_bb["flota"]["capacidad_pasajeros"]

cfg_escalon = cfg_sim["escalones"]["escalon_dia_10pct"]
hora_ini_min, hora_fin_min = cfg_escalon["horas"][0] * 60, cfg_escalon["horas"][1] * 60
FLOTAS = cfg_escalon["num_barcos_barrido"]
SEMILLAS_EVAL = cfg_sim["agente"]["evaluacion"]["semillas"]
DEMANDA_DIR = BASE / "politica_base" / "output" / "escalon_dia_10pct"

print(f"Flotas: {FLOTAS}")
print(f"Semillas de evaluacion: {SEMILLAS_EVAL}")
print(f"Capacidad: {capacidad}  Horizonte: [{hora_ini_min},{hora_fin_min}] min")

Flotas: [4, 6, 8, 10, 12, 14, 16]
Semillas de evaluacion: [1001, 1002, 1003, 1004, 1005]
Capacidad: 30  Horizonte: [360,1440] min


In [2]:
POLITICAS_CON_RESERVA = {"H1": asignar_flota_h1, "H2": asignar_flota_h2, "H3": asignar_flota_h3}

def correr(grupos_df, num_barcos, nombre_politica):
    env = SimuladorBarcosBergen(
        grupos_df=grupos_df, matriz_tiempos=matriz_tiempos, nodos=nodos,
        num_barcos=num_barcos, capacidad_barco=capacidad,
        nodo_inicial=cfg_bb["flota"]["nodo_inicial"], paso_tiempo_min=cfg_sim["paso_tiempo_min"],
        hora_inicio_min=hora_ini_min, hora_fin_min=hora_fin_min,
        cfg_recompensa=cfg_sim["recompensa"], unidad_demanda=cfg_sim["unidad_demanda"],
    )
    obs, info = env.reset(seed=cfg_sim["semilla"])
    reservas = {}
    while True:
        estado = info["_estado_obj"]
        libres = [b for b in estado.barcos if b.libre]
        if nombre_politica == "H0":
            decisiones = asignar_flota(libres, estado, matriz_tiempos, capacidad, cfg_sim["recompensa"])
        else:
            decisiones, reservas = POLITICAS_CON_RESERVA[nombre_politica](
                libres, estado, matriz_tiempos, capacidad, reservas, cfg_sim["recompensa"])
        accion = np.array([env.codificar_accion_barco(decisiones[b.id]) if b.libre else 0 for b in estado.barcos])
        obs, r, terminated, truncated, info = env.step(accion)
        if truncated or terminated:
            break
    return env


def fila_metricas(env, politica, num_barcos, semilla):
    cons = met.verificar_conservacion(env)
    glob = met.metricas_globales(env)
    usr = met.metricas_por_usuario(env)
    por_barco = met.metricas_por_barco(env)
    backlog = met.sin_atender_al_final_por_par(env)
    movs = [e for e in env.log_eventos if e["tipo"] == "movimiento"]
    movimientos_vacios = sum(1 for m in movs if m["ocupacion"] == 0)
    viajes = list(met._tiempos_viaje_por_unidad(env).values())
    return {
        "politica": politica, "num_barcos": num_barcos, "semilla": semilla,
        "conservacion_cuadra": cons["cuadra"], "pct_atendidas": glob["pct_atendidas"],
        "backlog_final_total": int(backlog["sin_atender"].sum()) if len(backlog) else 0,
        "espera_media_min": glob["espera_media_min"], "espera_p95_min": usr["espera_min"]["p95"],
        "espera_max_min": usr["espera_min"]["max"],
        "viaje_medio_min": float(np.mean(viajes)) if viajes else float("nan"),
        "sistema_medio_min": glob["sistema_medio_min"],
        "movimientos_totales": int(por_barco["movimientos"].sum()),
        "movimientos_vacios": movimientos_vacios, "movimientos_con_carga": len(movs) - movimientos_vacios,
        "ocupacion_pct_capacidad": float(por_barco["ocupacion_media"].mean()) / capacidad * 100.0,
        "pct_esperando_flota": float(por_barco["pct_esperando"].mean()),
    }


def serie_movimiento(env):
    filas = []
    for frame in env.historial_estados:
        n_moviendo = sum(1 for b in frame["barcos"] if not b["libre"])
        filas.append({"minuto_del_dia": frame["tiempo"]["minuto_del_dia"], "n_moviendo": n_moviendo,
                      "n_quieto": len(frame["barcos"]) - n_moviendo})
    return pd.DataFrame(filas)

In [3]:
grupos_por_semilla = {s: pd.read_csv(DEMANDA_DIR / f"grupos_seed{s}.csv") for s in SEMILLAS_EVAL}
semilla_series = SEMILLAS_EVAL[0]  # unica semilla para la serie minuto a minuto

filas = []
series_por_flota = {}  # (politica, num_barcos) -> DataFrame, solo para semilla_series
t0 = time.time()
total = 4 * len(FLOTAS) * len(SEMILLAS_EVAL)
i = 0
for semilla, grupos in grupos_por_semilla.items():
    for num_barcos in FLOTAS:
        for politica in ["H0", "H1", "H2", "H3"]:
            i += 1
            env = correr(grupos, num_barcos, politica)
            filas.append(fila_metricas(env, politica, num_barcos, semilla))
            if semilla == semilla_series:
                series_por_flota[(politica, num_barcos)] = serie_movimiento(env)
    print(f"  semilla {semilla} lista ({i}/{total}, {time.time()-t0:.0f}s)")

resultados = pd.DataFrame(filas)
assert resultados["conservacion_cuadra"].all()
print(f"\n{len(resultados)} corridas, conservacion OK, {time.time()-t0:.0f}s")

resultados.to_csv(BASE / "heuristicas" / "outputs" / "resultados" / "resultados_h0_h1_h2_h3.csv", index=False)
cols = [c for c in resultados.columns if c not in ("politica", "num_barcos", "semilla", "conservacion_cuadra")]
resumen = resultados.groupby(["politica", "num_barcos"])[cols].agg(["mean", "std"])
resumen.columns = [f"{c}_{s}" for c, s in resumen.columns]
resumen = resumen.reset_index()
resumen.to_csv(BASE / "heuristicas" / "outputs" / "resultados" / "resumen_h0_h1_h2_h3.csv", index=False)
resumen[["politica", "num_barcos", "espera_media_min_mean", "espera_p95_min_mean", "espera_max_min_mean", "pct_atendidas_mean"]]

  semilla 1001 lista (28/140, 13s)
  semilla 1002 lista (56/140, 30s)
  semilla 1003 lista (84/140, 51s)
  semilla 1004 lista (112/140, 73s)
  semilla 1005 lista (140/140, 92s)

140 corridas, conservacion OK, 92s


,politica,num_barcos,espera_media_min_mean,espera_p95_min_mean,espera_max_min_mean,pct_atendidas_mean
0,H0,4,73.947701,197.9380,241.194,99.365422
1,H0,6,26.567529,77.1855,98.858,99.391710
2,H0,8,12.027839,29.6976,43.236,99.476859
3,H0,10,7.615098,17.9720,28.382,99.481843
4,H0,12,6.178374,15.7058,28.026,99.583395
5,H0,14,5.612886,14.8100,25.958,99.480642
6,H0,16,5.475918,14.7780,25.466,99.456656
7,H1,4,71.465781,196.4340,241.410,99.404409
8,H1,6,25.380392,73.0901,95.934,99.384170
9,H1,8,12.010102,29.9840,46.394,99.362004


## Curva de servicio -- 9 métricas, interactiva

In [4]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

politicas_orden = ["H0", "H1", "H2", "H3"]
COLOR = {"H0": "#2a78d6", "H1": "#eb6834", "H2": "#1baf7a", "H3": "#008300"}
DASH = {"H0": "dash", "H1": "dot", "H2": "dashdot", "H3": "solid"}

paneles = [
    ("espera_media_min", "Espera media (min)"),
    ("espera_p95_min", "Espera P95 (min)"),
    ("espera_max_min", "Espera máxima (min)"),
    ("sistema_medio_min", "Tiempo en sistema medio (min)"),
    ("viaje_medio_min", "Tiempo de viaje medio (min)"),
    ("backlog_final_total", "Backlog al cierre (personas)"),
    ("movimientos_totales", "Movimientos totales"),
    ("ocupacion_pct_capacidad", "Ocupación media (% de capacidad)"),
    ("pct_esperando_flota", "% tiempo la flota está ociosa"),
]

fig = make_subplots(rows=3, cols=3, subplot_titles=[p[1] for p in paneles], vertical_spacing=0.09)
for idx, (col, titulo) in enumerate(paneles):
    row, coln = idx // 3 + 1, idx % 3 + 1
    for pol in politicas_orden:
        sub = resumen[resumen["politica"] == pol].sort_values("num_barcos")
        fig.add_trace(go.Scatter(
            x=sub["num_barcos"], y=sub[f"{col}_mean"],
            error_y=dict(type="data", array=sub[f"{col}_std"], visible=True, thickness=1),
            mode="lines+markers", name=pol, line=dict(color=COLOR[pol], dash=DASH[pol]),
            legendgroup=pol, showlegend=(idx == 0),
            hovertemplate=f"{pol}<br>%{{x}} barcos<br>{titulo}: %{{y:.2f}}<extra></extra>",
        ), row=row, col=coln)
    fig.update_xaxes(title_text="Nº barcos", row=row, col=coln, tickmode="array", tickvals=FLOTAS)

fig.update_layout(height=1050, width=1150, template="plotly_white",
                   title="H0 vs H1 vs H2 vs H3 -- curva de servicio, flota 4-16 (media ± std, 5 semillas)")
fig.show()

## ¿Se satura la curva en 14-16 barcos?

In [5]:
tabla_resumen = resumen[resumen["politica"] == "H3"][["num_barcos", "espera_media_min_mean", "espera_p95_min_mean"]].sort_values("num_barcos")
tabla_resumen["delta_pct"] = tabla_resumen["espera_media_min_mean"].pct_change() * 100
tabla_resumen.rename(columns={"espera_media_min_mean": "espera_media_H3", "espera_p95_min_mean": "espera_p95_H3"})

,num_barcos,espera_media_H3,espera_p95_H3,delta_pct
21,4,72.923893,169.6600,NaN
22,6,24.324092,64.4140,-66.644551
23,8,10.639184,26.3055,-56.260714
24,10,6.304722,15.9460,-40.740548
25,12,4.811801,12.2680,-23.679418
26,14,4.389071,11.6474,-8.785274
27,16,4.085065,11.2260,-6.926433


## Barcos en movimiento vs. quietos, por hora del día

Para H3 (la política recomendada), en las 7 flotas -- muestra si, con más
barcos, buena parte de la flota queda ociosa fuera de la hora pico (lo que
sugeriría que se podría operar con menos barcos en esos momentos, y sumar
más solo en punta).

In [6]:
import plotly.express as px

fig2 = go.Figure()
paleta = px.colors.sequential.Viridis
for i, nb in enumerate(FLOTAS):
    sub = series_por_flota[("H3", nb)].copy()
    sub["hora"] = sub["minuto_del_dia"] / 60
    sub["pct_moviendo"] = sub["n_moviendo"] / nb * 100
    color = paleta[int(i / max(1, len(FLOTAS) - 1) * (len(paleta) - 1))]
    fig2.add_trace(go.Scatter(
        x=sub["hora"], y=sub["pct_moviendo"], mode="lines", name=f"{nb} barcos", line=dict(color=color),
        hovertemplate=f"{nb} barcos<br>%{{x:.1f}}h<br>%{{y:.0f}}%% en movimiento<extra></extra>",
    ))
fig2.update_layout(title="H3 -- % de la flota en movimiento por hora del día, según tamaño de flota",
                    xaxis_title="Hora del día", yaxis_title="% de la flota en movimiento",
                    template="plotly_white", height=500)
fig2.show()

In [7]:
# Mismo minuto a minuto, pero a UN tamaño de flota fijo, comparando las 4 politicas
NUM_BARCOS_FIJO = 12
fig3 = go.Figure()
for pol in politicas_orden:
    sub = series_por_flota[(pol, NUM_BARCOS_FIJO)].copy()
    sub["hora"] = sub["minuto_del_dia"] / 60
    sub["pct_moviendo"] = sub["n_moviendo"] / NUM_BARCOS_FIJO * 100
    fig3.add_trace(go.Scatter(x=sub["hora"], y=sub["pct_moviendo"], mode="lines", name=pol,
                              line=dict(color=COLOR[pol], dash=DASH[pol]),
                              hovertemplate=f"{pol}<br>%{{x:.1f}}h<br>%{{y:.0f}}%% en movimiento<extra></extra>"))
fig3.update_layout(title=f"H0 vs H1 vs H2 vs H3 -- % de la flota en movimiento por hora, {NUM_BARCOS_FIJO} barcos",
                    xaxis_title="Hora del día", yaxis_title="% de la flota en movimiento",
                    template="plotly_white", height=500)
fig3.show()

## Interpretación

- **Rendimientos decrecientes claros**, y con 14-16 barcos la curva de H3 ya
  está prácticamente plana -- la tabla de "delta %" de arriba lo muestra
  numéricamente: cada barco adicional aporta cada vez menos.
- **La gráfica de % en movimiento por hora confirma la intuición operativa**:
  con flotas grandes (14-16), gran parte de los barcos está ociosa fuera de
  las horas de más demanda -- exactamente el patrón que sugeriría operar con
  una flota variable (más barcos activos en punta, menos en los valles) en
  vez de una flota fija todo el día. Este notebook no implementa esa
  estrategia (activar/desactivar barcos dinámicamente sería un cambio de
  arquitectura, no una heurística de despacho) -- pero la evidencia para
  justificarla como línea de trabajo futura queda aquí, medida.
- Comparando políticas a flota fija (12 barcos): H3 mantiene mayor % de
  tiempo ocioso que H0 en general, pero de forma más selectiva -- consistente
  con el hallazgo de `03` anterior (menos movimientos, mejor targeteados).